In [1]:
import pandas as pd

df=pd.read_csv('100_Unique_QA_Dataset.csv')

In [2]:
def tokenize(text):
    text=text.lower()
    text=text.replace("?","")
    text=text.replace("'","")
    return text.split()

In [3]:
tokenize("What is the capital city of France ?")

['what', 'is', 'the', 'capital', 'city', 'of', 'france']

In [4]:
vocab={"<UNK>":0}

In [5]:
def build_vocab(row):
    tokenized_question=tokenize(row['question'])
    tokenized_answer=tokenize(row['answer'])

    merged_tokens=tokenized_question+tokenized_answer

    for token in merged_tokens:

        if token not in vocab:
            vocab[token]=len(vocab)

In [6]:
df.apply(build_vocab,axis=1)

0     None
1     None
2     None
3     None
4     None
      ... 
85    None
86    None
87    None
88    None
89    None
Length: 90, dtype: object

In [7]:
len(vocab)

324

In [8]:
def text_to_indices(text,vocab):

    indexed_text=[]

    for token in tokenize(text):

        if token in vocab:
            indexed_text.append(vocab[token])
        else:
            indexed_text.append(vocab['<UNK>'])

    return indexed_text

In [9]:
import torch
from torch.utils.data import DataLoader,Dataset

In [11]:
class QADataset(Dataset):
    def __init__(self,df,vocab):
        self.df=df
        self.vocab=vocab

    def __len__(self):
        return self.df.shape[0]

    def __getitem__(self,index):
        numerical_question=text_to_indices(self.df.iloc[index]['question'],self.vocab)
        numerical_answer=text_to_indices(self.df.iloc[index]['answer'],self.vocab)
        return torch.tensor(numerical_question),torch.tensor(numerical_answer)

In [12]:
dataset=QADataset(df,vocab)
dataloader=DataLoader(dataset,batch_size=1,shuffle=True)

In [13]:
import torch.nn as nn

In [25]:
class SimpleRNN(nn.Module):

    def __init__(self,vocab_size):
        super().__init__()
        self.embedding=nn.Embedding(vocab_size,embedding_dim=50)
        self.rnn=nn.RNN(50,64,batch_first=True)
        self.fc=nn.Linear(64,vocab_size)

    def forward(self,question):
        embedding_question=self.embedding(question)
        hidden,final=self.rnn(embedding_question)
        output=self.fc(final.squeeze(0))
        return output

In [26]:
lr=0.001
epochs=20

In [27]:
model=SimpleRNN(len(vocab))
criterion = nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=lr)

In [29]:
for epoch in range(epochs):
    total_loss=0
    for question,answer in dataloader:

        optimizer.zero_grad()
        output=model(question)
        loss=criterion(output,answer[0])
        loss.backward()
        optimizer.step()
        total_loss+=loss.item()

    print(f"Epoch {epoch+1}, Loss:{total_loss:4f}")

Epoch 1, Loss:522.982054
Epoch 2, Loss:455.095565
Epoch 3, Loss:375.105773
Epoch 4, Loss:311.966915
Epoch 5, Loss:260.676843
Epoch 6, Loss:212.999618
Epoch 7, Loss:169.722663
Epoch 8, Loss:132.537792
Epoch 9, Loss:102.042845
Epoch 10, Loss:78.202164
Epoch 11, Loss:60.814519
Epoch 12, Loss:48.164578
Epoch 13, Loss:37.925178
Epoch 14, Loss:30.869626
Epoch 15, Loss:25.614162
Epoch 16, Loss:21.579226
Epoch 17, Loss:18.089552
Epoch 18, Loss:15.594861
Epoch 19, Loss:13.823202
Epoch 20, Loss:12.045920


In [33]:
def predict(model,question,threshold=0.5):
    numeric_question=text_to_indices(question,vocab)
    question_tensor=torch.tensor(numeric_question).unsqueeze(0)
    output=model(question_tensor)
    prob=nn.functional.softmax(output,dim=1)
    value,index=torch.max(prob,dim=1)

    if value<threshold:
        print("I don't know")

    print(list(vocab.keys())[index])

In [34]:
predict(model,"What is the largest planet in our solar system")

jupiter
